## WISER CAPEX Study

In [ ]:
from pathlib import Path
import sys

# --- paths & imports ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent   # 
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"


# --- simulation config -------------------------------------------------------

# Toggle which modules to run, dashboards, result collection, etc.
# (Adjust flags / keys to match your setup if they differ.)
sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": True, # set to false to disable the dashboard
    "run_opex": True,
    "opex_dashboard": True, # set to false to disable the dashboard
    "run_lifetime_extension": False,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": True, # set to false to disable the dashboard
    "collect_results": True,
}

# --- parameter space (unchanged) ---------------------------------------------

parameter_space = {
    # Flattened: CAPEX_overrides -> Capex_inputFiles -> PA
    "CAPEX_overrides.Capex_inputFiles.PA": [
        "CAPEX"
        "CAPEX_OpenDoor.yaml",
        "CAPEX_IncreasedBarriers.yaml",
        "CAPEX_EconomicDownturn.yaml",
        "CAPEX_GlobalEscalation.yaml",
    ],

    # Same for materials
    "CAPEX_overrides.Material_inputFiles.MD": [
        "Commoditz_Params"
        "Commodity_Params_OpenDoor.yaml",
        "Commodity_Params_IncreasedBarriers.yaml",
        "Commodity_Params_EconomicDownturn.yaml",
        "Commodity_Params_GlobalEscalation.yaml",
    ],

    # Optional scenario label                                                                                               
    "Scenario.name": [
        "Open Door",
        "Increased Barriers",
        "Economic Downturn",
        "Global Escalation",
    ],

    "FINEX_overrides.WACC.flag_fixed_WACC": [True, True, True, True],
    "FINEX_overrides.WACC.WACC_annual":     [0.072, 0.072, 0.072, 0.072],
}





# --- zip groups: bind CAPEX + Material (+ Scenario.name) together ------------

zip_groups = {
    "macro_scenarios": [
        "CAPEX_overrides.Capex_inputFiles.PA",
        "CAPEX_overrides.Material_inputFiles.MD",
        "Scenario.name",
        "FINEX_overrides.WACC.flag_fixed_WACC",
        "FINEX_overrides.WACC.WACC_annual",
    ]
}

# --- build + run experiment -----------------------------------------------

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",        
    simulation_config=sim_cfg,             
    parameter_space=parameter_space,       
    base_seed=42,                          
    replicates=1, # This are the Monte Carlo replicates per param set. We can run this on the cluster for more samples.                        
    name="CAPEX_Uncertainty",              
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,                 
)

# Run the full sweep – returns a DataFrame of run metadata

exp.run()


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)


[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



## WISER OPEX Study

### Curtailment Overrides

In [3]:
from pathlib import Path
import sys

# --- paths & imports ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent   # 
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"


# --- simulation config -------------------------------------------------------

# Toggle which modules to run, dashboards, result collection, etc.
# (Adjust flags / keys to match your setup if they differ.)
sim_cfg = {
        "run_marketenv": True,
        "run_metenv": False,
        "run_capex": True,
        "capex_dashboard": False,
        "run_curtailment": True,
        "run_opex": True,
        "opex_dashboard": False,
        "run_lifetime_extension": False,
        "run_revenue": True,
        "run_valuation": True,
        "valuation_dashboard": True,
        "collect_results": True,
    }

parameter_space = {
    # -------------------------------
    # High-level curtailment switch
    # -------------------------------
    "Curtailment_overrides.curt_input.Curtailment.apply_curtailment": (
        False,  # C0 — Reference (no curtailment at all)
        True,   # C1
        True,   # C2
        True,   # C3
        True,   # C4
        True,   # C5
    ),

    # -------------------------------
    # Explicit uncertainty activation
    # -------------------------------
    "Curtailment_overrides.curt_input.Curtailment.reduceProduction.apply_epistemic_uncertainty": (
        False,  # C0
        True,   # C1
        True,   # C2
        True,   # C3
        True,   # C4
        True,   # C5
    ),

    "Curtailment_overrides.curt_input.Curtailment.reduceProduction.apply_aleatory_uncertainty": (
        False,  # C0
        True,   # C1
        True,   # C2
        True,   # C3
        True,   # C4
        True,   # C5
    ),

    # -------------------------------
    # Gamma parameters (fractions)
    # -------------------------------
    "Curtailment_overrides.curt_input.Curtailment.reduceProduction.gamma_shape": (
        (1.0, 1.0),        # C0 — Reference
        (0.414, 0.506),    # C1 — Low transmission constraints
        (0.747, 0.913),    # C2 — Medium transmission constraints
        (1.395, 1.705),    # C3 — High transmission constraints
        (0.567, 0.693),    # C4 — Very high market curtailment
        (0.351, 0.429),    # C5 — Storage solutions
    ),

    "Curtailment_overrides.curt_input.Curtailment.reduceProduction.gamma_scale": (
        (0.0, 0.0),                    # C0 — Reference
        (0.01503, 0.01837),             # C1 — 1.503–1.837 %
        (0.04014, 0.04906),             # C2 — 4.014–4.906 %
        (0.05994, 0.07326),             # C3 — 5.994–7.326 %
        (0.04653, 0.05687),             # C4 — 4.653–5.687 %
        (0.01620, 0.01980),             # C5 — 1.62–1.98 %
    ),

    # -------------------------------
    # Scenario names
    # -------------------------------
    "Scenario.name": (
        "C0 — Reference (no curtailment)",
        "C1 — Low transmission constraints",
        "C2 — Medium transmission constraints",
        "C3 — High transmission constraints",
        "C4 — Very high market curtailment occurrence",
        "C5 — Storage solutions development",
    ),
}

zip_groups = {
    "curtailment_scenarios": [
        "Curtailment_overrides.curt_input.Curtailment.apply_curtailment",
        "Curtailment_overrides.curt_input.Curtailment.reduceProduction.apply_epistemic_uncertainty",
        "Curtailment_overrides.curt_input.Curtailment.reduceProduction.apply_aleatory_uncertainty",
        "Curtailment_overrides.curt_input.Curtailment.reduceProduction.gamma_shape",
        "Curtailment_overrides.curt_input.Curtailment.reduceProduction.gamma_scale",
        "Scenario.name",
    ]
}


# --- build + run experiment -----------------------------------------------

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",        
    simulation_config=sim_cfg,             
    parameter_space=parameter_space,       
    base_seed=42,                          
    replicates=1,                        
    name="Curtailment_Uncertainty",              
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,                 
)

# Run the full sweep – returns a DataFrame of run metadata

exp.run()




m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)


[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



KeyboardInterrupt: 

### LTE Overrides

In [ ]:
from pathlib import Path
import sys

# --- paths & imports ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent   # 
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"


# --- simulation config -------------------------------------------------------

# Toggle which modules to run, dashboards, result collection, etc.
# (Adjust flags / keys to match your setup if they differ.)
sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": False,
    "run_opex": True,
    "opex_dashboard": False,
    "run_lifetime_extension": True,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": True,
    "collect_results": True,
}

parameter_space = {
    # Human-readable label
    "Scenario.name": [
        "Baseline – No LTE",
        "EOL1 – Moderate degradation",
        "EOL2 – Substantial degradation",
    ],

    # Enable / disable LTE
    "LTE_overrides.lte_input.LTE.apply_lte": [
        False,
        True,
        True,
    ],

    # Extension horizon (hours)
    # Baseline: keep extension_h=0 to avoid any accidental extension logic
    "LTE_overrides.lte_input.LTE.extension_h": [
        0,
        43000,
        43000,
    ],

    # ----------------------------
    # AEP haircut distribution (fractional)
    # ----------------------------
    "LTE_overrides.lte_input.LTE.aep_haircut.mu": [
        0.0,
        -0.020,   # moderate AEP loss (=-2.0%)
        -0.040,   # substantial AEP loss (=-4.0%)
    ],
    "LTE_overrides.lte_input.LTE.aep_haircut.sigma": [
        0.0,
        0.010,
        0.015,
    ],
    # Keep bounds conservative and code-consistent
    "LTE_overrides.lte_input.LTE.aep_haircut.min": [
        0.0,     # irrelevant when sigma=0, but keep consistent
        -0.30,
        -0.30,
    ],
    "LTE_overrides.lte_input.LTE.aep_haircut.max": [
        0.0,
        0.0,
        0.0,
    ],

    # ----------------------------
    # Reliability / OPEX mean-shift factors (extension regime only)
    # ----------------------------
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.flag_apply": [
        False,
        True,
        True,
    ],
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.lambda_factor": [
        1.00,
        1.20,
        1.35,
    ],
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.mttr_factor": [
        1.00,
        1.10,
        1.20,
    ],
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.mttwL_factor": [
        1.00,
        1.10,
        1.20,
    ],
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.tau_factor": [
        1.00,
        1.00,
        1.00,
    ],
    "LTE_overrides.lte_input.LTE.opex_extension.analytic_ctmc.mean_shift.p_access_factor": [
        1.00,
        0.90,
        0.80,
    ],

    # ----------------------------
    # Refurbishment uplift distribution (€/turbine)
    # NOTE: LTE.py samples refurb_uplift and multiplies by n_turbines
    # ----------------------------
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.dist": [
        "fixed",        # baseline
        "normal_trunc",  # EOL1
        "normal_trunc",  # EOL2
    ],
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.value": [
        0.0,    # only used when dist=fixed
        None,
        None,
    ],
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.mu": [
        0.0,        # baseline (unused if dist=fixed, but safe)
        1_000_000,  # moderate uplift
        3_000_000,  # substantial uplift
    ],
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.sigma": [
        0.0,
        400_000,
        800_000,
    ],
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.min": [
        0.0,
        0.0,
        0.0,
    ],
    "LTE_overrides.lte_input.LTE.costs.refurb_uplift.max": [
        0.0,     # baseline (unused if fixed)
        None,    # let it float or set a cap if you prefer
        None,
    ],
}


# Zip everything so we get exactly 3 coherent scenarios (not a Cartesian product)
zip_groups = {
    "lte_scenarios": list(parameter_space.keys())
}


# --- build + run experiment -----------------------------------------------

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",        
    simulation_config=sim_cfg,             
    parameter_space=parameter_space,       
    base_seed=42,                          
    replicates=3,                        
    name="LTE_Experiment",              
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,                 
)

# Run the full sweep – returns a DataFrame of run metadata

exp.run()




m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)


[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


...

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



[prices] first=2017-03-31 00:00:00  last=2041-11-24 15:00:00.000000001  count=216113
[power ] first=2017-03-31 00:00:00  last=2041-11-24 15:00:00.000000001  count=216113
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### Reliability overrides

In [ ]:
from pathlib import Path
import sys

# --- paths & imports ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"


# --- simulation config -------------------------------------------------------

sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": False,
    "run_opex": True,
    "opex_dashboard": False,
    "run_lifetime_extension": False,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": False,
    "collect_results": True,
}

parameter_space = {
    # ------------------------------------------------------------------
    # Ensure the uncertainty block is actually executed in OPEX.py:
    # (In your code, uncertainty sampling is coupled to mean_shift.flag_apply.)
    # ------------------------------------------------------------------
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.flag_apply": [
        True,  # Reference
        True,  # R1
        True,  # R2
        True,  # R3
    ],

    # ------------------------------------------------------------------
    # Mean-shift factors (α_λ = 1.00 for all scenarios in your LaTeX table)
    # ------------------------------------------------------------------
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.lambda_factor": [
        1.00,  # Reference
        1.00,  # R1
        1.00,  # R2
        1.00,  # R3
    ],

    # Keep other mean shifts neutral (optional, but helps reproducibility)
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttr_factor": [
        1.00, 1.00, 1.00, 1.00
    ],
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttwL_factor": [
        1.00, 1.00, 1.00, 1.00
    ],
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.tau_factor": [
        1.00, 1.00, 1.00, 1.00
    ],

    # ------------------------------------------------------------------
    # Epistemic uncertainty on λ: Gamma CV per your LaTeX table
    # ------------------------------------------------------------------
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_epistemic_lambda": [
        False,  # Reference: deterministic λ (CV=0)
        True,   # R1
        True,   # R2
        True,   # R3
    ],

    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.lambda_gamma_cv": [
        0.00,  # Reference
        0.15,  # R1 – Low epistemic uncertainty
        0.30,  # R2 – Mid epistemic uncertainty
        0.60,  # R3 – High epistemic uncertainty
    ],

    # ------------------------------------------------------------------
    # Turn OFF process uncertainty (since your table is only about λ)
    # ------------------------------------------------------------------
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_process": [
        False, False, False, False
    ],
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttr_sigma": [
        0.00, 0.00, 0.00, 0.00
    ],
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttwL_sigma": [
        0.00, 0.00, 0.00, 0.00
    ],

    # ------------------------------------------------------------------
    # Scenario labels
    # ------------------------------------------------------------------
    "Scenario.name": [
        "Reference",
        "R1 – Low epistemic uncertainty",
        "R2 – Mid epistemic uncertainty",
        "R3 – High epistemic uncertainty",
    ],
}

zip_groups = {
    "failure_rate_uncertainty_scenarios": [
        # mean-shift controls (kept neutral but required to enable sampling in your code)
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.flag_apply",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.lambda_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttr_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttwL_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.tau_factor",

        # epistemic λ uncertainty
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_epistemic_lambda",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.lambda_gamma_cv",

        # process uncertainty (off)
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_process",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttr_sigma",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttwL_sigma",

        # label
        "Scenario.name",
    ]
}


# --- build + run experiment --------------------------------------------------

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",
    simulation_config=sim_cfg,
    parameter_space=parameter_space,
    base_seed=42,
    replicates=1,
    name="OPEX_Reliability_Uncertainty",
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,
)

exp.run()


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)


[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old beha

[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old beha

### OPEC Inefficiencies


In [2]:
from pathlib import Path
import sys

# --- paths & imports ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from core.Simulation import build_experiment

LIBRARY_PATH = PROJECT_ROOT / "examples" / "Inputs" / "HKN"
RESULT_DIR   = PROJECT_ROOT / "results"


# ---------------------------------------------------------------------
# Simulation config (from notebook)
# ---------------------------------------------------------------------
sim_cfg = {
    "run_marketenv": True,
    "run_metenv": False,
    "run_capex": True,
    "capex_dashboard": False,
    "run_opex": True,
    "opex_dashboard": False,
    "run_lifetime_extension": False,
    "run_revenue": True,
    "run_valuation": True,
    "valuation_dashboard": False,
    "collect_results": True,
}

# ---------------------------------------------------------------------
# Design of Experiments (from notebook)
# ---------------------------------------------------------------------

parameter_space = {
    # Mean-shift factors
    # Failure rates fixed at baseline
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.lambda_factor": [
        1.00, 1.00, 1.00, 1.00, 1.00
    ],
    # MTTR alpha
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttr_factor": [
        1.00, 1.00, 1.00, 0.95, 0.80
    ],
    # MTTW_L alpha
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttwL_factor": [
        1.00, 1.00, 1.00, 0.90, 0.75
    ],
    # Access factor alpha_p
    "OPEX_overrides.parameters.analytic_ctmc.mean_shift.p_access_factor": [
        1.00, 0.90, 1.00, 1.05, 1.10
    ],

    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_process": [
        False, True, True, True, True
    ],  


    # Uncertainty
    # Failure-rate uncertainty disabled (fixed baseline failure rates)
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.lamda_sigma": [
        0.00, 0.00, 0.00, 0.00, 0.00
    ],
    # MTTR sigma
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttr_sigma": [
        0.00, 0.55, 0.35, 0.25, 0.18
    ],
    # MTTW_L sigma
    "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttwL_sigma": [
        0.00, 0.85, 0.50, 0.30, 0.25
    ],

    # Scenario labels
    "Scenario.name": [
        "P0 -- Reference",
        "P1 -- High process uncertainty",
        "P2 -- Typical offshore operations",
        "P3 -- Mature O&M processes",
        "P4 -- Best practice O&M",
    ],
}

zip_groups = {
    "macro_scenarios": [
        # mean shifts
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.lambda_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttr_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.mttwL_factor",
        "OPEX_overrides.parameters.analytic_ctmc.mean_shift.p_access_factor",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.flag_apply_process",
        # uncertainties
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.lamda_sigma",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttr_sigma",
        "OPEX_overrides.parameters.analytic_ctmc.uncertainty.mttwL_sigma",
        # label
        "Scenario.name",
    ]
}


# --- build + run experiment --------------------------------------------------

exp = build_experiment(
    library_path=str(LIBRARY_PATH),
    base_config_path="Config.yaml",
    simulation_config=sim_cfg,
    parameter_space=parameter_space,
    base_seed=42,
    replicates=1,
    name="OPEX_Efficiency",
    result_directory=str(RESULT_DIR),
    zip_groups=zip_groups,
)

exp.run()


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.cost_records = pd.concat([self.cost_records, pd.DataFrame(rows)], ignore_index=True)


[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old beha

[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old beha

...

[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\CAPEX.py:505: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old beha

[prices] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
[power ] first=2017-03-31 00:00:00  last=2037-03-25 23:00:00  count=175200
Timestamps identical (length & order): True


m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:237: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:243: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample(rule)
m:\Projects\Paper_Projects\NAWEA_Study\Price_indices\WINPACT\winpact\core\Revenue_Model.py:334: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.groupby(pd.Grouper(key="settlement_time", freq=pay_freq))


,scenario_id,label,seed,status,duration_s,error_message,traceback,experiment_name,result_directory
0,5a6a633829,lambda_factor=1.0__mttr_factor=1.0__mttwL_fact...,138436968,success,4.041192,None,None,OPEX_Efficiency,m:\Projects\Paper_Projects\NAWEA_Study\Price_i...
1,a7856012ea,lambda_factor=1.0__mttr_factor=1.0__mttwL_fact...,402480696,success,80.520821,None,None,OPEX_Efficiency,m:\Projects\Paper_Projects\NAWEA_Study\Price_i...
2,c2ba94dc79,lambda_factor=1.0__mttr_factor=1.0__mttwL_fact...,275608486,success,32.016475,None,None,OPEX_Efficiency,m:\Projects\Paper_Projects\NAWEA_Study\Price_i...
3,f4a89f13aa,lambda_factor=1.0__mttr_factor=0.95__mttwL_fac...,1716874378,success,3.829509,None,None,OPEX_Efficiency,m:\Projects\Paper_Projects\NAWEA_Study\Price_i...
4,95998e1b12,lambda_factor=1.0__mttr_factor=0.8__mttwL_fact...,916830209,success,3.982267,None,None,OPEX_Efficiency,m:\Projects\Paper_Projects\NAWEA_Study\Price_i...
